# Pseudo-label DL detector -- offline submission notebook
**Internet must be disabled.** Loads the trained weights from a local path (attached Kaggle Dataset input), runs sliding-window heatmap detection + your already-tuned classical tracker on every test dataset, writes and validates `submission.csv`.

**Before running:** attach your trained-weights dataset, set `MODEL_PATH` below.

In [1]:
# NOTE: internet must be DISABLED for this notebook. Every import is a package already
# installed in the Kaggle image -- only the trained weights are read from a local path
# (attached as a Kaggle Dataset input), not downloaded.
import os, json, gc
from dataclasses import dataclass
from collections import defaultdict
from typing import Tuple, List

import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment
from scipy.ndimage import gaussian_filter, maximum_filter

import torch
import torch.nn as nn
import torch.nn.functional as F

print("device available:", "cuda" if torch.cuda.is_available() else "cpu")


device available: cuda


In [2]:
# Physical voxel scale (z, y, x) in micrometres per voxel
SCALE = np.array([1.625, 0.40625, 0.40625], dtype=np.float64)

@dataclass
class ImageVolume:
    path: str
    shape: tuple
    dtype: np.dtype
    chunk: tuple

    @property
    def n_t(self) -> int:
        return int(self.shape[0])

    def frame(self, t: int) -> np.ndarray:
        return _read_chunk(self.path, t, self.shape, self.dtype)

def open_image(zarr_path: str) -> ImageVolume:
    with open(os.path.join(zarr_path, "0", "zarr.json")) as f:
        meta = json.load(f)
    shape = tuple(int(s) for s in meta["shape"])
    dtype = np.dtype(meta["data_type"])
    return ImageVolume(path=zarr_path, shape=shape, dtype=dtype, chunk=None)

def _read_chunk(zarr_path: str, t: int, shape: tuple, dtype: np.dtype) -> np.ndarray:
    """Read and decode one timepoint chunk -> (Z, Y, X)."""
    frame_shape = shape[1:]
    chunk_path = os.path.join(zarr_path, "0", "c", str(t), "0", "0", "0")
    try:
        import blosc2
        with open(chunk_path, "rb") as f:
            raw = f.read()
        dec = blosc2.decompress(raw)
        arr = np.frombuffer(dec, dtype=dtype)
        if arr.size == int(np.prod(frame_shape)):
            return arr.reshape(frame_shape).copy()
    except Exception:
        import zarr
        z = zarr.open(os.path.join(zarr_path, "0"), mode="r")
        return np.asarray(z[t])

@dataclass
class TrackGraph:
    node_t: np.ndarray
    node_z: np.ndarray
    node_y: np.ndarray
    node_x: np.ndarray
    node_ids: np.ndarray
    edges: np.ndarray
    meta: dict

    @property
    def n_nodes(self) -> int:
        return len(self.node_ids)

    @property
    def n_edges(self) -> int:
        return len(self.edges)

def _ball_footprint(radius_um: float, eff_spacing: np.ndarray) -> np.ndarray:
    rad_vox = np.maximum(1, np.round(radius_um / eff_spacing).astype(int))
    zz, yy, xx = np.ogrid[-rad_vox[0]:rad_vox[0]+1,
                          -rad_vox[1]:rad_vox[1]+1,
                          -rad_vox[2]:rad_vox[2]+1]
    d = ((zz * eff_spacing[0])**2 + (yy * eff_spacing[1])**2 + (xx * eff_spacing[2])**2)
    return d <= radius_um**2

def load_geff(geff_path: str) -> dict:
    """Read a .geff ground-truth graph (train only)."""
    import zarr
    z = zarr.open(geff_path, mode="r")
    return dict(
        node_ids=np.asarray(z["nodes/ids"]),
        t=np.asarray(z["nodes/props/t/values"]),
        z=np.asarray(z["nodes/props/z/values"]),
        y=np.asarray(z["nodes/props/y/values"]),
        x=np.asarray(z["nodes/props/x/values"]),
        edges=np.asarray(z["edges/ids"]),
    )

print("Data I/O loaded")


Data I/O loaded


### Model -- denser 3D U-Net (must match the architecture used for training)

In [3]:
import torch
import torch.nn as nn

class ResConvBlock(nn.Module):
    """Residual double-conv block -- helps gradient flow in a deeper network than the
    original 3-level LightUNet3D."""
    def __init__(self, cin, cout):
        super().__init__()
        self.conv1 = nn.Conv3d(cin, cout, 3, padding=1)
        self.norm1 = nn.InstanceNorm3d(cout)
        self.conv2 = nn.Conv3d(cout, cout, 3, padding=1)
        self.norm2 = nn.InstanceNorm3d(cout)
        self.act = nn.LeakyReLU(0.1, inplace=True)
        self.skip = nn.Conv3d(cin, cout, 1) if cin != cout else nn.Identity()

    def forward(self, x):
        identity = self.skip(x)
        out = self.act(self.norm1(self.conv1(x)))
        out = self.norm2(self.conv2(out))
        return self.act(out + identity)


class DenseUNet3D(nn.Module):
    """Wider and deeper than LightUNet3D: 4 levels instead of 3, residual blocks instead
    of plain double-conv, configurable base_ch (default 24 vs. the original 16).
    ~8-12M params depending on base_ch -- still trains reasonably fast, meaningfully more
    capacity than the ~1.4M-param original."""
    def __init__(self, base_ch=24):
        super().__init__()
        c1, c2, c3, c4, c5 = base_ch, base_ch*2, base_ch*4, base_ch*8, base_ch*16
        self.enc1 = ResConvBlock(1, c1)
        self.enc2 = ResConvBlock(c1, c2)
        self.enc3 = ResConvBlock(c2, c3)
        self.enc4 = ResConvBlock(c3, c4)
        self.bottleneck = ResConvBlock(c4, c5)

        self.pool = nn.MaxPool3d(2)

        self.up4 = nn.ConvTranspose3d(c5, c4, 2, stride=2)
        self.dec4 = ResConvBlock(c4 * 2, c4)
        self.up3 = nn.ConvTranspose3d(c4, c3, 2, stride=2)
        self.dec3 = ResConvBlock(c3 * 2, c3)
        self.up2 = nn.ConvTranspose3d(c3, c2, 2, stride=2)
        self.dec2 = ResConvBlock(c2 * 2, c2)
        self.up1 = nn.ConvTranspose3d(c2, c1, 2, stride=2)
        self.dec1 = ResConvBlock(c1 * 2, c1)

        self.head = nn.Conv3d(c1, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b = self.bottleneck(self.pool(e4))

        d4 = self._pad_cat(self.up4(b), e4)
        d4 = self.dec4(d4)
        d3 = self._pad_cat(self.up3(d4), e3)
        d3 = self.dec3(d3)
        d2 = self._pad_cat(self.up2(d3), e2)
        d2 = self.dec2(d2)
        d1 = self._pad_cat(self.up1(d2), e1)
        d1 = self.dec1(d1)

        return torch.sigmoid(self.head(d1))

    @staticmethod
    def _pad_cat(up, skip):
        import torch.nn.functional as F
        diffZ = skip.size(2) - up.size(2)
        diffY = skip.size(3) - up.size(3)
        diffX = skip.size(4) - up.size(4)
        up = F.pad(up, [diffX // 2, diffX - diffX // 2,
                         diffY // 2, diffY - diffY // 2,
                         diffZ // 2, diffZ - diffZ // 2])
        return torch.cat([up, skip], dim=1)


def count_params(m):
    return sum(p.numel() for p in m.parameters())



print('DenseUNet3D model module loaded')


DenseUNet3D model module loaded


### Peak extraction + sliding-window inference

In [4]:
import numpy as np
import torch

def make_gaussian_heatmap(shape, centers_zyx, sigma_z=1.2, sigma_xy=2.5):
    """Build a target heatmap with a 3D Gaussian bump at each center.
    shape: (Z,Y,X). centers_zyx: list/array of (z,y,x) voxel coords.
    Overlapping Gaussians are combined with elementwise max (not sum) so nearby
    cells don't create one inflated blob.
    """
    hm = np.zeros(shape, dtype=np.float32)
    if len(centers_zyx) == 0:
        return hm
    Z, Y, X = shape
    rz = max(1, int(round(sigma_z * 3)))
    ry = max(1, int(round(sigma_xy * 3)))
    rx = max(1, int(round(sigma_xy * 3)))
    for (cz, cy, cx) in centers_zyx:
        cz, cy, cx = float(cz), float(cy), float(cx)
        z0, z1 = max(0, int(cz - rz)), min(Z, int(cz + rz) + 1)
        y0, y1 = max(0, int(cy - ry)), min(Y, int(cy + ry) + 1)
        x0, x1 = max(0, int(cx - rx)), min(X, int(cx + rx) + 1)
        if z0 >= z1 or y0 >= y1 or x0 >= x1:
            continue
        zz, yy, xx = np.meshgrid(np.arange(z0, z1), np.arange(y0, y1), np.arange(x0, x1), indexing='ij')
        g = np.exp(-(((zz - cz) ** 2) / (2 * sigma_z ** 2) +
                      ((yy - cy) ** 2) / (2 * sigma_xy ** 2) +
                      ((xx - cx) ** 2) / (2 * sigma_xy ** 2)))
        hm[z0:z1, y0:y1, x0:x1] = np.maximum(hm[z0:z1, y0:y1, x0:x1], g)
    return hm


def centernet_focal_loss(pred, target, alpha=2.0, beta=4.0, eps=1e-6):
    """CenterNet-style penalty-reduced focal loss for heatmap regression.
    Pixels near a labeled center (target close to 1) are penalized less even if
    predicted low; pixels far from any label (target ~0) are penalized less as they
    approach 1 too, EXCEPT true background which is still pushed down. This matters
    here because annotations are SPARSE -- a plain per-pixel BCE would punish the
    model for correctly firing on real-but-unlabeled cells. This loss is softer
    about that than plain BCE, though it is not a full fix (see caveat in the
    training notebook markdown).
    """
    pred = pred.clamp(eps, 1 - eps)
    pos_mask = (target >= 0.98).float()
    neg_mask = (target < 0.98).float()

    pos_loss = -torch.log(pred) * torch.pow(1 - pred, alpha) * pos_mask
    neg_loss = -torch.log(1 - pred) * torch.pow(pred, alpha) * torch.pow(1 - target, beta) * neg_mask

    n_pos = pos_mask.sum()
    pos_loss = pos_loss.sum()
    neg_loss = neg_loss.sum()

    if n_pos == 0:
        return neg_loss
    return (pos_loss + neg_loss) / n_pos


def extract_peaks(heatmap, min_distance_vox=(2, 4, 4), threshold=0.3, max_peaks=2000):
    """Local-maxima peak extraction from a predicted heatmap -> (N,3) zyx coords + scores."""
    from scipy.ndimage import maximum_filter
    footprint_shape = tuple(2 * d + 1 for d in min_distance_vox)
    mx = maximum_filter(heatmap, size=footprint_shape, mode='nearest')
    peaks_mask = (heatmap == mx) & (heatmap >= threshold)
    coords = np.argwhere(peaks_mask)
    scores = heatmap[peaks_mask]
    if len(coords) > max_peaks:
        idx = np.argsort(scores)[::-1][:max_peaks]
        coords, scores = coords[idx], scores[idx]
    return coords.astype(np.float64), scores.astype(np.float64)

print('Peak extraction module loaded')


Peak extraction module loaded


In [5]:
import numpy as np
import torch

def sliding_window_infer(model, vol, patch=(32, 128, 128), overlap=0.25, device='cpu'):
    """Run model over a full volume in overlapping patches, average overlaps -> full heatmap.
    vol: (Z,Y,X) numpy array (already normalized to roughly [0,1]).
    """
    Z, Y, X = vol.shape
    pz, py, px = patch
    pz, py, px = min(pz, Z), min(py, Y), min(px, X)
    sz = max(1, int(pz * (1 - overlap)))
    sy = max(1, int(py * (1 - overlap)))
    sx = max(1, int(px * (1 - overlap)))

    z_starts = list(range(0, max(1, Z - pz + 1), sz))
    y_starts = list(range(0, max(1, Y - py + 1), sy))
    x_starts = list(range(0, max(1, X - px + 1), sx))
    if z_starts[-1] + pz < Z: z_starts.append(Z - pz)
    if y_starts[-1] + py < Y: y_starts.append(Y - py)
    if x_starts[-1] + px < X: x_starts.append(X - px)

    heat_sum = np.zeros((Z, Y, X), dtype=np.float32)
    weight = np.zeros((Z, Y, X), dtype=np.float32)

    model.eval()
    with torch.no_grad():
        for z0 in z_starts:
            for y0 in y_starts:
                for x0 in x_starts:
                    patch_vol = vol[z0:z0+pz, y0:y0+py, x0:x0+px]
                    t = torch.from_numpy(patch_vol)[None, None].float().to(device)
                    pred = model(t)[0, 0].cpu().numpy()
                    heat_sum[z0:z0+pz, y0:y0+py, x0:x0+px] += pred
                    weight[z0:z0+pz, y0:y0+py, x0:x0+px] += 1.0

    weight = np.maximum(weight, 1e-6)
    return heat_sum / weight

print('Sliding-window module loaded')


Sliding-window module loaded


### Classical tracker (reused, unchanged from your tuned baseline)

In [6]:
"""
Division-aware linking module for the Biohub cell tracking pipeline.

Drop-in replacement for `link_motion_enhanced`. Adds a second pass after the
standard 1-to-1 Hungarian assignment that looks for mitosis events: an
existing track whose predicted position is close to TWO next-frame
detections (instead of one) is treated as a division, and BOTH daughters
get an edge from the parent's last node. This is the piece your current
pipeline is missing entirely (it can only ever produce 1-to-1 edges).

Usage: replace the call to `link_motion_enhanced(...)` in `process_dataset`
with `link_motion_with_divisions(...)`. Same signature, extra kwargs at the
end with sane defaults.
"""

import numpy as np
from scipy.optimize import linear_sum_assignment
from collections import defaultdict

SCALE = np.array([1.625, 0.40625, 0.40625], dtype=np.float64)  # z, y, x um/voxel


class Track:
    __slots__ = ['pos', 'vel', 'node_id', 'miss', 'alive']
    def __init__(self, pos, node_id):
        self.pos = pos.copy()
        self.vel = np.zeros(3)
        self.node_id = node_id
        self.miss = 0
        self.alive = True


def link_motion_with_divisions(
    frames: list,
    max_link_um: float = 8.0,
    motion_weight: float = 0.7,
    max_miss: int = 2,
    # --- new division-related params ---
    division_radius_um: float = 6.0,     # how close a 2nd daughter candidate must be to the parent's predicted pos
    division_min_gap_um: float = 1.0,    # daughters must be at least this far apart from each other (else it's just noise/jitter)
    division_symmetry_tol: float = 0.6,  # max allowed relative difference in distance-from-parent between the two daughters (0=perfectly symmetric)
    max_daughters_per_parent: int = 2,
):
    """
    Frame-to-frame linking with basic mitosis detection.

    Strategy per transition t -> t+1:
      1. Standard Hungarian assignment on scaled centroid distance
         (same as before) gives each track its single best-match daughter.
      2. For every UNMATCHED detection in frame t+1, check every ALIVE
         track's predicted position. If the unmatched detection is within
         `division_radius_um` of a track that already got a match this
         frame, AND the two candidate daughters are roughly symmetric in
         distance from the parent (a coarse but decent proxy for "these
         two cells emerged from the same parent" vs "this is just a
         different nearby cell"), record a second edge from the parent's
         last node -> this detection. The track continues as BOTH children
         (division = branching, not termination), so we spawn a new Track
         object for the second daughter.
    """
    node_ids, node_t, node_z, node_y, node_x = [], [], [], [], []
    nid = 1
    frame_ids = []
    tracks = []

    for t, coords in enumerate(frames):
        ids_t = []
        for coord in coords:
            node_ids.append(nid)
            node_t.append(t)
            node_z.append(coord[0]); node_y.append(coord[1]); node_x.append(coord[2])
            ids_t.append(nid)
            if t == 0:
                tracks.append(Track(np.asarray(coord, dtype=np.float64), nid))
            nid += 1
        frame_ids.append(ids_t)

    def empty_graph():
        return dict(
            node_t=np.array(node_t, dtype=np.int64),
            node_z=np.array(node_z, dtype=np.float64),
            node_y=np.array(node_y, dtype=np.float64),
            node_x=np.array(node_x, dtype=np.float64),
            node_ids=np.array(node_ids, dtype=np.int64),
            edges=np.array([], dtype=np.int64).reshape(-1, 2),
        )

    if len(frames) <= 1:
        return empty_graph()

    edges = []

    for t in range(1, len(frames)):
        current_coords = frames[t]
        current_ids = frame_ids[t]

        alive = [tr for tr in tracks if tr.alive]

        if len(current_coords) == 0 or len(alive) == 0:
            for tr in alive:
                tr.miss += 1
                tr.pos = tr.pos + tr.vel
                if tr.miss > max_miss:
                    tr.alive = False
            continue

        cur = np.asarray(current_coords, dtype=np.float64)
        predicted = np.array([tr.pos + tr.vel for tr in alive])

        d_scaled = ((predicted[:, None, :] - cur[None, :, :]) * SCALE).astype(np.float64)
        dist = np.sqrt((d_scaled ** 2).sum(axis=2))  # (n_tracks, n_detections)

        cost = np.where(dist <= max_link_um, dist, 1e6)
        row_ind, col_ind = linear_sum_assignment(cost)

        matched_track_for_det = {}   # det_idx -> track_idx (primary match)
        used_dets = set()

        for r, c in zip(row_ind, col_ind):
            if dist[r, c] > max_link_um:
                continue
            tr = alive[r]
            new_pos = cur[c]
            edges.append((tr.node_id, current_ids[c]))
            tr.vel = motion_weight * (new_pos - tr.pos) + (1 - motion_weight) * tr.vel
            tr.pos = new_pos
            tr.node_id = current_ids[c]
            tr.miss = 0
            matched_track_for_det[c] = r
            used_dets.add(c)

        # unmatched tracks: predict forward, allow a few missed frames
        matched_track_rows = set(matched_track_for_det.values())
        for r, tr in enumerate(alive):
            if r not in matched_track_rows:
                tr.miss += 1
                tr.pos = tr.pos + tr.vel
                if tr.miss > max_miss:
                    tr.alive = False

        # --- Division pass: look for a 2nd daughter for tracks that already matched ---
        unmatched_dets = [c for c in range(len(cur)) if c not in used_dets]
        if unmatched_dets and matched_track_for_det:
            new_tracks = []
            daughter_count = defaultdict(lambda: 1)  # primary match already counts as 1

            for c in unmatched_dets:
                best_r, best_dist = None, None
                for c2, r in matched_track_for_det.items():
                    if daughter_count[r] >= max_daughters_per_parent:
                        continue
                    tr = alive[r]
                    d_parent = np.sqrt((((tr.pos - cur[c]) * SCALE) ** 2).sum())
                    if d_parent > division_radius_um:
                        continue
                    # symmetry check vs the primary daughter already assigned to this parent
                    d_primary = dist[r, c2]
                    if d_primary <= 1e-9:
                        continue
                    asym = abs(d_parent - d_primary) / max(d_parent, d_primary)
                    d_between = np.sqrt(((( cur[c2] - cur[c]) * SCALE) ** 2).sum())
                    if d_between < division_min_gap_um:
                        continue
                    if asym > division_symmetry_tol:
                        continue
                    if best_dist is None or d_parent < best_dist:
                        best_r, best_dist = r, d_parent

                if best_r is not None:
                    parent_tr = alive[best_r]
                    edges.append((parent_tr.node_id, current_ids[c]))
                    daughter_count[best_r] += 1
                    # spawn a new track for this second daughter
                    new_tr = Track(cur[c], current_ids[c])
                    new_tr.vel = parent_tr.vel.copy()
                    new_tracks.append(new_tr)
                    used_dets.add(c)

            tracks.extend(new_tracks)

        # any still-unmatched detections become brand new tracks (new cell entering FOV, etc.)
        for c in range(len(cur)):
            if c not in used_dets:
                tracks.append(Track(cur[c], current_ids[c]))

        tracks = [tr for tr in tracks if tr.alive] + [tr for tr in tracks if not tr.alive and tr not in tracks]
        # (keep dead tracks out; simpler: just filter alive, dead ones drop naturally next loop since we rebuild `alive` from `tracks`)
        tracks = [tr for tr in tracks if tr.alive]

    g = empty_graph()
    g["edges"] = np.array(edges, dtype=np.int64).reshape(-1, 2) if edges else np.array([], dtype=np.int64).reshape(-1, 2)
    return g

In [7]:
# ============ APPEARANCE-AWARE, DIVISION-PRESERVING LINKER ============
# Drop-in upgrade to link_motion_with_divisions. Same division-detection logic (unchanged),
# but the primary Hungarian assignment cost now also penalizes mismatched intensity/score
# between a track's predicted appearance and each candidate detection -- this disambiguates
# two similarly-positioned cells in dense regions where centroid distance alone is ambiguous.
# Needs per-frame scores (frame_scores[t][i] lines up with frames[t][i]); if you're not using
# detect_blobs_watershed, pass frame_scores = [np.ones(len(f)) for f in frames] to disable the
# appearance term (appearance_weight has no effect when all scores are equal).

class TrackV2:
    __slots__ = ['pos', 'vel', 'score', 'node_id', 'miss', 'alive']
    def __init__(self, pos, score, node_id):
        self.pos = pos.copy()
        self.vel = np.zeros(3)
        self.score = score
        self.node_id = node_id
        self.miss = 0
        self.alive = True


def link_motion_with_divisions_v2(
    frames: list,
    frame_scores: list,
    max_link_um: float = 8.0,
    motion_weight: float = 0.7,
    max_miss: int = 2,
    appearance_weight: float = 4.0,
    division_radius_um: float = 6.0,
    division_min_gap_um: float = 1.0,
    division_symmetry_tol: float = 0.6,
    max_daughters_per_parent: int = 2,
):
    node_ids, node_t, node_z, node_y, node_x = [], [], [], [], []
    nid = 1
    frame_ids = []
    tracks = []

    for t, coords in enumerate(frames):
        ids_t = []
        scores_t = frame_scores[t]
        for i, coord in enumerate(coords):
            node_ids.append(nid)
            node_t.append(t)
            node_z.append(coord[0]); node_y.append(coord[1]); node_x.append(coord[2])
            ids_t.append(nid)
            if t == 0:
                tracks.append(TrackV2(np.asarray(coord, dtype=np.float64), float(scores_t[i]), nid))
            nid += 1
        frame_ids.append(ids_t)

    def empty_graph():
        return dict(
            node_t=np.array(node_t, dtype=np.int64),
            node_z=np.array(node_z, dtype=np.float64),
            node_y=np.array(node_y, dtype=np.float64),
            node_x=np.array(node_x, dtype=np.float64),
            node_ids=np.array(node_ids, dtype=np.int64),
            edges=np.array([], dtype=np.int64).reshape(-1, 2),
        )

    if len(frames) <= 1:
        return empty_graph()

    # normalize the score scale once so appearance_weight is comparable across datasets
    nonempty = [np.asarray(s) for s in frame_scores if len(s)]
    all_scores_flat = np.concatenate(nonempty) if nonempty else np.array([1.0])
    score_scale = np.std(all_scores_flat) + 1e-6

    edges = []

    for t in range(1, len(frames)):
        current_coords = frames[t]
        current_scores = frame_scores[t]
        current_ids = frame_ids[t]

        alive = [tr for tr in tracks if tr.alive]

        if len(current_coords) == 0 or len(alive) == 0:
            for tr in alive:
                tr.miss += 1
                tr.pos = tr.pos + tr.vel
                if tr.miss > max_miss:
                    tr.alive = False
            continue

        cur = np.asarray(current_coords, dtype=np.float64)
        cur_scores = np.asarray(current_scores, dtype=np.float64)
        predicted = np.array([tr.pos + tr.vel for tr in alive])
        pred_scores = np.array([tr.score for tr in alive])

        d_scaled = ((predicted[:, None, :] - cur[None, :, :]) * SCALE).astype(np.float64)
        dist = np.sqrt((d_scaled ** 2).sum(axis=2))
        appearance_diff = np.abs(pred_scores[:, None] - cur_scores[None, :]) / score_scale

        cost = np.where(dist <= max_link_um, dist + appearance_weight * appearance_diff, 1e6)
        row_ind, col_ind = linear_sum_assignment(cost)

        matched_track_for_det = {}
        used_dets = set()

        for r, c in zip(row_ind, col_ind):
            if dist[r, c] > max_link_um:
                continue
            tr = alive[r]
            new_pos = cur[c]
            edges.append((tr.node_id, current_ids[c]))
            tr.vel = motion_weight * (new_pos - tr.pos) + (1 - motion_weight) * tr.vel
            tr.pos = new_pos
            tr.score = 0.7 * tr.score + 0.3 * cur_scores[c]
            tr.node_id = current_ids[c]
            tr.miss = 0
            matched_track_for_det[c] = r
            used_dets.add(c)

        matched_track_rows = set(matched_track_for_det.values())
        for r, tr in enumerate(alive):
            if r not in matched_track_rows:
                tr.miss += 1
                tr.pos = tr.pos + tr.vel
                if tr.miss > max_miss:
                    tr.alive = False

        # --- division pass: unchanged from link_motion_with_divisions ---
        unmatched_dets = [c for c in range(len(cur)) if c not in used_dets]
        if unmatched_dets and matched_track_for_det:
            new_tracks = []
            daughter_count = defaultdict(lambda: 1)

            for c in unmatched_dets:
                best_r, best_dist = None, None
                for c2, r in matched_track_for_det.items():
                    if daughter_count[r] >= max_daughters_per_parent:
                        continue
                    tr = alive[r]
                    d_parent = np.sqrt((((tr.pos - cur[c]) * SCALE) ** 2).sum())
                    if d_parent > division_radius_um:
                        continue
                    d_primary = dist[r, c2]
                    if d_primary <= 1e-9:
                        continue
                    asym = abs(d_parent - d_primary) / max(d_parent, d_primary)
                    d_between = np.sqrt(((( cur[c2] - cur[c]) * SCALE) ** 2).sum())
                    if d_between < division_min_gap_um:
                        continue
                    if asym > division_symmetry_tol:
                        continue
                    if best_dist is None or d_parent < best_dist:
                        best_r, best_dist = r, d_parent

                if best_r is not None:
                    parent_tr = alive[best_r]
                    edges.append((parent_tr.node_id, current_ids[c]))
                    daughter_count[best_r] += 1
                    new_tr = TrackV2(cur[c], cur_scores[c], current_ids[c])
                    new_tr.vel = parent_tr.vel.copy()
                    new_tracks.append(new_tr)
                    used_dets.add(c)

            tracks.extend(new_tracks)

        for c in range(len(cur)):
            if c not in used_dets:
                tracks.append(TrackV2(cur[c], cur_scores[c], current_ids[c]))

        tracks = [tr for tr in tracks if tr.alive]

    g = empty_graph()
    g["edges"] = np.array(edges, dtype=np.int64).reshape(-1, 2) if edges else np.array([], dtype=np.int64).reshape(-1, 2)
    return g

print("Appearance-aware linking module loaded")


Appearance-aware linking module loaded


In [8]:
from scipy.ndimage import gaussian_filter1d

def close_gaps_enhanced(frames: list, g: TrackGraph, max_gap: int = 2,
                       gap_dist_um: float = 8.0) -> TrackGraph:
    """Enhanced gap closing with interpolation."""
    if g.n_edges == 0:
        return g
    
    coords = {int(nid): (int(g.node_t[i]), g.node_z[i], g.node_y[i], g.node_x[i])
              for i, nid in enumerate(g.node_ids)}
    
    has_out = set(int(s) for s, _ in g.edges)
    has_in = set(int(t) for _, t in g.edges)
    
    ends_by_t = defaultdict(list)
    starts_by_t = defaultdict(list)
    
    for nid, (t, z, y, x) in coords.items():
        if nid not in has_out:
            ends_by_t[t].append(nid)
        if nid not in has_in:
            starts_by_t[t].append(nid)
    
    new_nodes = []
    new_edges = []
    next_id = int(g.node_ids.max()) + 1 if g.n_nodes else 1
    
    for gap in range(1, max_gap + 1):
        for t, ends in ends_by_t.items():
            starts = starts_by_t.get(t + gap + 1, [])
            if not starts:
                continue
            
            ec = np.array([[coords[e][1], coords[e][2], coords[e][3]] for e in ends]) * SCALE
            sc = np.array([[coords[s][1], coords[s][2], coords[s][3]] for s in starts]) * SCALE
            
            if len(ec) == 0 or len(sc) == 0:
                continue
            
            d = np.sqrt(((ec[:, None, :] - sc[None, :, :])**2).sum(axis=2))
            thr = gap_dist_um * (gap + 1)
            cost = np.where(d <= thr, d, 1e6)
            
            row_ind, col_ind = linear_sum_assignment(cost)
            used_s = set()
            
            for r, c in zip(row_ind, col_ind):
                if d[r, c] > thr or ends[r] in has_out or starts[c] in used_s:
                    continue
                
                e_id, s_id = ends[r], starts[c]
                te, ze, ye, xe = coords[e_id]
                ts, zs, ys, xs = coords[s_id]
                
                prev = e_id
                for k in range(1, gap + 1):
                    frac = k / (gap + 1)
                    zi = ze + (zs - ze) * frac
                    yi = ye + (ys - ye) * frac
                    xi = xe + (xs - xe) * frac
                    nid = next_id
                    next_id += 1
                    new_nodes.append((te + k, zi, yi, xi, nid))
                    new_edges.append((prev, nid))
                    prev = nid
                new_edges.append((prev, s_id))
                has_out.add(e_id)
                used_s.add(c)
    
    if not new_nodes:
        return g
    
    nt = np.concatenate([g.node_t, np.array([n[0] for n in new_nodes], dtype=np.int64)])
    nz = np.concatenate([g.node_z, np.array([n[1] for n in new_nodes])])
    ny = np.concatenate([g.node_y, np.array([n[2] for n in new_nodes])])
    nx = np.concatenate([g.node_x, np.array([n[3] for n in new_nodes])])
    nid = np.concatenate([g.node_ids, np.array([n[4] for n in new_nodes], dtype=np.int64)])
    edges = np.concatenate([g.edges, np.array(new_edges, dtype=np.int64).reshape(-1, 2)])
    
    return TrackGraph(node_t=nt, node_z=nz, node_y=ny, node_x=nx, 
                     node_ids=nid, edges=edges, meta=g.meta)

def prune_isolated(g: TrackGraph) -> TrackGraph:
    """Remove nodes not referenced by any edge."""
    if g.n_edges == 0:
        return g
    
    used = set(int(x) for x in g.edges.reshape(-1))
    keep = np.array([i for i, nid in enumerate(g.node_ids) if int(nid) in used])
    
    if len(keep) == len(g.node_ids):
        return g
    
    return TrackGraph(
        node_t=g.node_t[keep], node_z=g.node_z[keep], 
        node_y=g.node_y[keep], node_x=g.node_x[keep],
        node_ids=g.node_ids[keep], edges=g.edges, meta=g.meta
    )

def smooth_tracks(g: TrackGraph) -> TrackGraph:
    """Smooth track positions using temporal filtering."""
    if g.n_nodes < 5:
        return g
    
    # Build adjacency
    adj = defaultdict(list)
    for src, tgt in g.edges:
        adj[src].append(tgt)
        adj[tgt].append(src)
    
    # Find tracks (connected components)
    visited = set()
    tracks = []
    
    for node in g.node_ids:
        if node in visited:
            continue
        stack = [node]
        track = []
        while stack:
            curr = stack.pop()
            if curr in visited:
                continue
            visited.add(curr)
            track.append(curr)
            for neighbor in adj[curr]:
                if neighbor not in visited:
                    stack.append(neighbor)
        if len(track) >= 5:
            tracks.append(track)
    
    # Smooth each track
    for track in tracks:
        # Get positions and times
        positions = []
        times = []
        for nid in track:
            idx = np.where(g.node_ids == nid)[0][0]
            positions.append([g.node_z[idx], g.node_y[idx], g.node_x[idx]])
            times.append(g.node_t[idx])
        
        positions = np.array(positions)
        
        # Check if times are sequential
        if len(np.unique(times)) == len(times):
            # Smooth positions
            smoothed = gaussian_filter1d(positions, sigma=0.5, axis=0, mode='nearest')
            
            # Update positions if not too far
            for i, nid in enumerate(track):
                idx = np.where(g.node_ids == nid)[0][0]
                dist = np.linalg.norm((positions[i] - smoothed[i]) * SCALE)
                if dist < 3.0:
                    g.node_z[idx] = smoothed[i, 0]
                    g.node_y[idx] = smoothed[i, 1]
                    g.node_x[idx] = smoothed[i, 2]
    
    return g

print("Post-processing module loaded")

Post-processing module loaded


### DL detection + classical tracking, combined

In [9]:
# ============ DL detection + classical tracker (combined) ============

def detect_frame_dl(model, vol, device, patch=(32, 128, 128), overlap=0.25,
                     min_distance_vox=(2, 5, 5), threshold=0.5, max_peaks=25000):
    lo, hi = np.percentile(vol, [1, 99.5])
    if hi <= lo: hi = lo + 1.0
    vol_norm = np.clip((vol.astype(np.float32) - lo) / (hi - lo), 0, 1).astype(np.float32)
    hm = sliding_window_infer(model, vol_norm, patch=patch, overlap=overlap, device=device)
    coords, scores = extract_peaks(hm, min_distance_vox=min_distance_vox, threshold=threshold, max_peaks=max_peaks)
    return coords, scores

def process_dataset_dl(zarr_path: str, model, device, **kwargs) -> TrackGraph:
    patch = kwargs.get('patch', (32, 128, 128))
    overlap = kwargs.get('overlap', 0.25)
    min_distance_vox = kwargs.get('min_distance_vox', (2, 5, 5))
    threshold = kwargs.get('threshold', 0.5)
    max_peaks = kwargs.get('max_peaks', 25000)

    max_link_um = kwargs.get('max_link_um', 6.5)
    motion_weight = kwargs.get('motion_weight', 0.85)
    max_miss = kwargs.get('max_miss', 1)
    use_appearance_cost = kwargs.get('use_appearance_cost', True)
    appearance_weight = kwargs.get('appearance_weight', 4.0)

    division_radius_um = kwargs.get('division_radius_um', 5.0)
    division_min_gap_um = kwargs.get('division_min_gap_um', 2.0)
    division_symmetry_tol = kwargs.get('division_symmetry_tol', 0.4)
    max_daughters_per_parent = kwargs.get('max_daughters_per_parent', 2)

    close_gaps = kwargs.get('close_gaps', True)
    max_gap = kwargs.get('max_gap', 1)
    gap_dist_um = kwargs.get('gap_dist_um', 5.0)
    smooth = kwargs.get('smooth', True)

    vol_meta = open_image(zarr_path)
    frames, frames_scores = [], []
    for t in range(vol_meta.n_t):
        vol = vol_meta.frame(t)
        coords, scores = detect_frame_dl(model, vol, device, patch=patch, overlap=overlap,
                                          min_distance_vox=min_distance_vox, threshold=threshold, max_peaks=max_peaks)
        frames.append(coords); frames_scores.append(scores)
        del vol
        if t % 10 == 0: gc.collect()

    if use_appearance_cost:
        g_dict = link_motion_with_divisions_v2(
            frames, frames_scores, max_link_um=max_link_um, motion_weight=motion_weight, max_miss=max_miss,
            appearance_weight=appearance_weight, division_radius_um=division_radius_um,
            division_min_gap_um=division_min_gap_um, division_symmetry_tol=division_symmetry_tol,
            max_daughters_per_parent=max_daughters_per_parent)
    else:
        g_dict = link_motion_with_divisions(
            frames, max_link_um=max_link_um, motion_weight=motion_weight, max_miss=max_miss,
            division_radius_um=division_radius_um, division_min_gap_um=division_min_gap_um,
            division_symmetry_tol=division_symmetry_tol, max_daughters_per_parent=max_daughters_per_parent)

    g = TrackGraph(**g_dict, meta={})
    if close_gaps:
        g = close_gaps_enhanced(frames, g, max_gap=max_gap, gap_dist_um=gap_dist_um)
    g = prune_isolated(g)
    if smooth:
        g = smooth_tracks(g)
    return g

print("DL process_dataset loaded")


DL process_dataset loaded


### Submission formatting + validator

In [10]:
def graph_to_rows(name: str, g: TrackGraph) -> list:
    """Convert graph to submission rows."""
    rows = []
    
    # Node rows
    for i in range(g.n_nodes):
        rows.append({
            "dataset": name,
            "row_type": "node",
            "node_id": int(g.node_ids[i]),
            "t": int(g.node_t[i]),
            "z": int(round(g.node_z[i])),
            "y": int(round(g.node_y[i])),
            "x": int(round(g.node_x[i])),
            "source_id": -1,
            "target_id": -1,
        })
    
    # Edge rows
    for src, tgt in g.edges:
        rows.append({
            "dataset": name,
            "row_type": "edge",
            "node_id": -1,
            "t": -1,
            "z": -1,
            "y": -1,
            "x": -1,
            "source_id": int(src),
            "target_id": int(tgt),
        })
    
    return rows

def create_submission(results: dict, output_path: str) -> pd.DataFrame:
    """Create submission file."""
    all_rows = []
    for name, g in results.items():
        all_rows.extend(graph_to_rows(name, g))
    
    df = pd.DataFrame(all_rows, columns=[
        "dataset", "row_type", "node_id", "t", "z", "y", "x", "source_id", "target_id"
    ])
    df.index.name = "id"
    df.to_csv(output_path)
    return df

print("Submission generator loaded")

Submission generator loaded


In [11]:
import pandas as pd
import numpy as np
import os

REQUIRED_COLS = ["id", "dataset", "row_type", "node_id", "t", "z", "y", "x", "source_id", "target_id"]

def validate_submission(csv_path, test_dir, expect_shape=None, verbose=True):
    """Validate a submission.csv before you trust it. Returns (ok: bool, report: dict).
    Checks:
      - required columns present, id is consecutive 0..N-1
      - row_type only 'node' or 'edge'
      - node rows: unique node_id per dataset, coords in-bounds (if expect_shape given),
        source_id/target_id == -1
      - edge rows: source_id/target_id reference existing node_ids in the SAME dataset,
        node_id/t/z/y/x == -1
      - every dataset folder name under test_dir (minus .zarr) appears in the submission
      - no dataset has zero nodes (silently empty prediction) or zero edges when it has >=2 frames worth of nodes
    """
    problems = []
    warnings = []
    df = pd.read_csv(csv_path)

    missing_cols = [c for c in REQUIRED_COLS if c not in df.columns]
    if missing_cols:
        problems.append(f"missing required columns: {missing_cols}")
        return False, {"problems": problems, "warnings": warnings}

    if not (df["id"].values == np.arange(len(df))).all():
        problems.append("`id` column is not consecutive integers starting at 0")

    bad_row_types = set(df["row_type"].unique()) - {"node", "edge"}
    if bad_row_types:
        problems.append(f"unexpected row_type values: {bad_row_types}")

    test_datasets = set()
    if test_dir and os.path.isdir(test_dir):
        test_datasets = {d[:-5] for d in os.listdir(test_dir) if d.endswith(".zarr")}
    submitted_datasets = set(df["dataset"].unique())
    missing_datasets = test_datasets - submitted_datasets
    extra_datasets = submitted_datasets - test_datasets
    if missing_datasets:
        problems.append(f"{len(missing_datasets)} test dataset(s) missing from submission: "
                         f"{sorted(missing_datasets)[:10]}{'...' if len(missing_datasets) > 10 else ''}")
    if extra_datasets:
        warnings.append(f"submission has dataset names not present in test_dir: {sorted(extra_datasets)[:10]}")

    per_dataset_stats = {}
    for ds, sub in df.groupby("dataset"):
        nodes = sub[sub["row_type"] == "node"]
        edges = sub[sub["row_type"] == "edge"]

        if nodes["node_id"].duplicated().any():
            problems.append(f"[{ds}] duplicate node_id values")

        if not (nodes["source_id"] == -1).all() or not (nodes["target_id"] == -1).all():
            problems.append(f"[{ds}] node rows must have source_id=target_id=-1")

        if not ((edges["node_id"] == -1).all() and (edges["t"] == -1).all() and
                (edges["z"] == -1).all() and (edges["y"] == -1).all() and (edges["x"] == -1).all()):
            problems.append(f"[{ds}] edge rows must have node_id=t=z=y=x=-1")

        valid_ids = set(nodes["node_id"].values.tolist())
        bad_src = ~edges["source_id"].isin(valid_ids)
        bad_tgt = ~edges["target_id"].isin(valid_ids)
        if bad_src.any() or bad_tgt.any():
            n_bad = int((bad_src | bad_tgt).sum())
            problems.append(f"[{ds}] {n_bad} edge(s) reference a node_id not present in this dataset's node rows")

        if expect_shape is not None:
            Z, Y, X = expect_shape
            oob = nodes[(nodes["z"] < 0) | (nodes["z"] >= Z) |
                        (nodes["y"] < 0) | (nodes["y"] >= Y) |
                        (nodes["x"] < 0) | (nodes["x"] >= X)]
            if len(oob):
                problems.append(f"[{ds}] {len(oob)} node(s) with out-of-bounds coordinates")

        if len(nodes) == 0:
            warnings.append(f"[{ds}] zero predicted nodes -- likely a detection failure for this dataset")
        if len(nodes) > 0 and len(edges) == 0:
            warnings.append(f"[{ds}] nodes present but zero edges -- linking may have failed")

        per_dataset_stats[ds] = dict(n_nodes=len(nodes), n_edges=len(edges))

    ok = len(problems) == 0
    if verbose:
        print(f"Validation {'PASSED' if ok else 'FAILED'} -- {len(problems)} problem(s), {len(warnings)} warning(s)")
        for p in problems:
            print("  [PROBLEM]", p)
        for w in warnings:
            print("  [WARNING]", w)
        n_ds = len(per_dataset_stats)
        if n_ds:
            avg_nodes = np.mean([s["n_nodes"] for s in per_dataset_stats.values()])
            avg_edges = np.mean([s["n_edges"] for s in per_dataset_stats.values()])
            print(f"  {n_ds} datasets, avg {avg_nodes:.1f} nodes/dataset, avg {avg_edges:.1f} edges/dataset")

    return ok, {"problems": problems, "warnings": warnings, "per_dataset": per_dataset_stats}

print('Validator loaded')


Validator loaded


### Config -- load weights offline

In [12]:
# ============ CONFIG -- load weights offline ============
MODEL_PATH = "/kaggle/input/datasets/mdaliazad/hybrid/pseudo_model_best(1).pt"  # <-- change this

INFER_CONFIG = dict(
    patch=(32, 128, 128), overlap=0.25, min_distance_vox=(2, 5, 5),
    threshold=0.5,   # calibrate against a train sample before trusting this
    max_peaks=25000,
    max_link_um=6.5, motion_weight=0.85, max_miss=1,
    use_appearance_cost=True, appearance_weight=4.0,
    division_radius_um=5.0, division_min_gap_um=2.0, division_symmetry_tol=0.4, max_daughters_per_parent=2,
    close_gaps=True, max_gap=1, gap_dist_um=5.0, smooth=True,
)

def find_test_dir():
    candidates = [
        "/kaggle/input/biohub-cell-tracking-during-development/test",
        "/kaggle/input/competitions/biohub-cell-tracking-during-development/test",
        os.environ.get("TEST_DIR", ""),
    ]
    for c in candidates:
        if c and os.path.isdir(c): return c
    for root, dirs, files in os.walk("/kaggle/input"):
        if os.path.basename(root) == "test":
            if any(d.endswith(".zarr") for d in dirs): return root
    raise FileNotFoundError("Test directory not found")

device = "cuda" if torch.cuda.is_available() else "cpu"
checkpoint = torch.load(MODEL_PATH, map_location=device, weights_only=False)
model = DenseUNet3D(base_ch=checkpoint.get("base_ch", 24)).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print(f"Loaded model from {MODEL_PATH} (epoch {checkpoint.get('epoch')}, loss {checkpoint.get('loss')})")


Loaded model from /kaggle/input/datasets/mdaliazad/hybrid/pseudo_model_best(1).pt (epoch 71, loss 0.540114293495814)


### Run inference on all test datasets

In [13]:
# ============ MAIN: run on all test datasets, write + validate submission.csv ============
TEST_DIR = find_test_dir()
print("test dir:", TEST_DIR)
dataset_names = sorted(d[:-5] for d in os.listdir(TEST_DIR) if d.endswith(".zarr"))
print(f"{len(dataset_names)} test datasets")

all_rows = []
for i, name in enumerate(dataset_names):
    print(f"[{i+1}/{len(dataset_names)}] {name}")
    zarr_path = os.path.join(TEST_DIR, name + ".zarr")
    with torch.no_grad():
        g = process_dataset_dl(zarr_path, model, device, **INFER_CONFIG)
    rows = graph_to_rows(name, g)
    all_rows.extend(rows)
    print(f"    nodes={g.n_nodes}  edges={g.n_edges}")
    gc.collect()
    if device == "cuda": torch.cuda.empty_cache()

for i, r in enumerate(all_rows):
    r["id"] = i
df = pd.DataFrame(all_rows, columns=["id","dataset","row_type","node_id","t","z","y","x","source_id","target_id"])
SUBMISSION_PATH = "/kaggle/working/submission.csv"
df.to_csv(SUBMISSION_PATH, index=False)
print(f"wrote {len(df)} rows to {SUBMISSION_PATH}")


test dir: /kaggle/input/competitions/biohub-cell-tracking-during-development/test
4 test datasets
[1/4] 44b6_0113de3b
    nodes=19360  edges=18217
[2/4] 44b6_0b24845f
    nodes=11062  edges=9275
[3/4] 6bba_05b6850b
    nodes=3758  edges=3518
[4/4] 6bba_05db0fb1
    nodes=40281  edges=36683
wrote 142154 rows to /kaggle/working/submission.csv


### Validate before submitting

In [14]:
# ============ VALIDATE before you trust it ============
ok, report = validate_submission(SUBMISSION_PATH, TEST_DIR, expect_shape=None, verbose=True)
if ok:
    print("\nsubmission.csv passed all structural checks.")
else:
    print("\nsubmission.csv has problems -- fix these before submitting:")
    for p in report["problems"]:
        print(" -", p)


Validation PASSED -- 0 problem(s), 0 warning(s)
  4 datasets, avg 18615.2 nodes/dataset, avg 16923.2 edges/dataset

submission.csv passed all structural checks.
